# DL 1, Pytorch

Привет! Этот ноутбук собран по мотивам семинара и сделан, чтобы ты получил опыт работы с PyTorch, GPU и получением градиентов.

В конце ты напишешь нейросеть для распознавания чисел!

**Для этого ноутбука надо включить GPU runtime, поэтому используйте https://colab.research.google.com/**.

In [ ]:
!nvidia-smi

In [ ]:
import torch

In [ ]:
torch.sq

In [ ]:
torch.*Tensor?

## Simple Pytorch

Изучим простые методы pytorch. Они сильно напоминают numpy, и часто можно использовать имя метода из numpy в pytorch.

In [ ]:
t = torch.Tensor(2, 3, 4)

In [ ]:
t

In [ ]:
t.shape

In [ ]:
t.random_(10)
t

In [ ]:
t = torch.zeros_like(t)
t

In [ ]:
r = torch.Tensor(t)
r.resize_(3, 8)
r

In [ ]:
a, b = torch.rand(1, 4), torch.rand(1, 4)
a + b

In [ ]:
a * b

In [ ]:
a @ b.T

In [ ]:
a, b = torch.rand(3, 4), torch.rand(4, 5)
a @ b

In [ ]:
a.norm(), b.sum()

In [ ]:
a / 10

In [ ]:
a.transpose(0, 1)

In [ ]:
t = torch.arange(10)
t.dtype

In [ ]:
t = t.to(torch.float32)
t, t.dtype

## Autograd & GPU

Теперь поработаем с GPU и получением градиентов.

In [ ]:
a, b = torch.rand(3, 4), torch.rand(4, 5)
a

In [ ]:
a = a.to("cuda")

In [ ]:
a @ b

In [ ]:
b = b.to("cuda")

In [ ]:
a @ b

In [ ]:
a.requires_grad_(True)

s = (a @ b).sum()
s

In [ ]:
s.backward()
s

In [ ]:
a, b

In [ ]:
a.grad, b.grad

In [ ]:
a, b = torch.rand(3, 4, requires_grad=True), torch.rand(4, 5, requires_grad=True)
s = (a @ b).sum()
s.backward()
a,b

In [ ]:
a, b

## Neural Network


Давайте определим простую ML-задачу и попробуем решить её с помощью нейросети.

In [ ]:
X = torch.rand(1000, 10)
w_true = torch.rand(10, 1) * 10
b_true = torch.tensor(3.1415926)
eps = torch.rand(1000) * 1e-3
y = X @ w_true + b_true + eps

In [ ]:
w = torch.rand(10, 1, requires_grad=True)
b = torch.rand(1, requires_grad=True)

In [ ]:
y_hat = X @ w + b
L = ((y_hat - y) ** 2).mean()
L

In [ ]:
L.backward()

In [ ]:
lr = 1e-2

with torch.no_grad():
    w -= w.grad * lr
    b -= b.grad * lr

In [ ]:
from tqdm import tqdm # Я решил сделать tqdm а то не красиво

In [ ]:
pbar = tqdm(range(5000))
for idx in pbar:
    w.grad = None
    b.grad = None
    y_hat = X @ w + b
    L = ((y_hat - y) ** 2).mean()
    L.backward()
    with torch.no_grad():
        w -= w.grad * lr
        b -= b.grad * lr
    pbar.set_postfix(loss=L.item())

In [ ]:
torch.norm(w_true - w), torch.norm(b_true - b)

In [ ]:
w_true.tolist(), w.tolist()

In [ ]:
b_true, b

Воспользуемся высокоуровневым способ описывать нейросети в PyTorch:

In [ ]:
class Linear(torch.nn.Module):
    def __init__(self, in_shape, out_shape):
        super().__init__()

        self.layer = torch.nn.Linear(in_shape, out_shape)

    def forward(self, x):
        return self.layer(x)

In [ ]:
model = Linear(10, 1)
optimizer = torch.optim.SGD(model.parameters(), lr)
criterion = torch.nn.MSELoss()

In [ ]:
model = model.to("cuda")

In [ ]:
pbar = tqdm(range(5000))
for idx in pbar:
    optimizer.zero_grad()
    y_hat = model(X.to("cuda")) # optimize!
    L = criterion(y_hat, y.to("cuda"))
    L.backward()
    optimizer.step()
    pbar.set_postfix(loss=L.item())

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
model_weight = list(model.parameters())
param = [f'w_{i}' for i in range(len(w_true))] + ['b']
w_true_np = w_true.cpu().detach().numpy().reshape(-1)
b_true_np = b_true.cpu().detach().numpy().reshape(-1)
w_np = model_weight[0].cpu().detach().numpy().reshape(-1)
b_np = model_weight[1].cpu().detach().numpy().reshape(-1)
df = pd.DataFrame({'Параметры': param,
                    'Истинное значение': np.concat((w_true_np, b_true_np)),
                    'Полученное значение': np.concat((w_np, b_np))
                    })
df['Разница'] = np.abs(df['Истинное значение'] - df['Полученное значение'])
df.head(11)

## MNIST

Перейдем к другой задаче -- распознавании чисел. Загрузим датасет MNIST и напишем нейросеть, которая отличает числа меньше 5 и больше или равно 5.

In [ ]:
from keras.datasets import mnist


(train_X, train_y), (test_X, test_y) = mnist.load_data()

In [ ]:
# print("\n".join(" ".join("X" if ch > 128 else "." for ch in row) for row in train_X[1].tolist())) у меня другой шрифт так что буду делать как ниже
import matplotlib.pyplot as plt
plt.imshow(train_X[1])
plt.show()

Создадим датасет для нашей задачи:

In [ ]:
train_X[1].shape

In [ ]:
(torch.tensor([1, 4, 5, 6, 7, 2]) >= 5).to(torch.long)

In [ ]:
class MNISTDataSet(torch.utils.data.Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X).float() / 255.0
        self.y = (torch.tensor(y) >= 5).to(torch.long)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

Опишем простую двухслойную сеть:

In [ ]:
class NNClassifier(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = torch.nn.Sequential(
            torch.nn.Flatten(),
            torch.nn.Linear(28 * 28, 128),
            torch.nn.ReLU()
        )
        self.layer2 = torch.nn.Linear(128, 2)
    def forward(self, x):
        return self.layer2(self.layer1(x))

Напишем для неё Loss-функцию

(подсказка: вспомните logistic regression)

In [ ]:
a = np.array([[1, 2, 3],
             [4, 5, 6],
             [7, 8, 9]])
b = np.array([0, 2, 1])
a[range(3), b]

In [ ]:
class NLLLoss(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.soft_max = torch.nn.Softmax(dim=1)
    def forward(self, y_pred, y):
        y_pred = torch.log(self.soft_max(y_pred) + 1e-9)
        L = -y_pred[:, 0] * (1 - y) - y_pred[:, 1] * y
        return L.mean()

Повторим цикл обучения для новой сети!

In [ ]:
!pip install comet_ml

In [ ]:
import comet_ml

exp = comet_ml.Experiment(api_key='comet_api_key', project_name="super-krutoi-NN")

In [ ]:
hyperparams = {
    "learning_rate": 0.001,
    "batch_size": 64,
    "epoch": 5,
    "optimizer": "Adam",
    "loss_function": "NLLLoss",
    "model": "NNClassifier (Flatten, Linear(784,128), ReLU, Linear(128,2))"
}
exp.log_parameters(hyperparams)

In [ ]:
train_dataset = MNISTDataSet(train_X, train_y)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=hyperparams["batch_size"], shuffle=True)

test_dataset = MNISTDataSet(test_X, test_y)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=hyperparams["batch_size"], shuffle=False)

model = NNClassifier().to('cuda')
criterion = NLLLoss().to('cuda')
optimizer = torch.optim.Adam(model.parameters(), lr=hyperparams["learning_rate"])


for i in tqdm(range(hyperparams["epoch"])):
    train_loss_sum = 0
    model.train()
    pbar = tqdm(train_loader, desc=f'Эпоха трейна: {i}')
    for x_batch, y_batch in pbar:
        model.zero_grad()
        x_batch = x_batch.to('cuda')
        y_batch = y_batch.to('cuda')
        y_pred = model(x_batch)
        loss = criterion(y_pred, y_batch)
        train_loss_sum += loss.item() * x_batch.size(0)
        loss.backward()
        optimizer.step()
    test_loss_sum = 0
    correct = 0
    model.eval()
    with torch.no_grad():
        pbar2 = tqdm(test_loader, desc=f'Эпоха теста: {i}')
        for x_batch, y_batch in pbar2:
            x_batch = x_batch.to('cuda')
            y_batch = y_batch.to('cuda')
            y_pred = model(x_batch)
            loss = criterion(y_pred, y_batch)
            test_loss_sum += loss.item() * x_batch.size(0)
            pred = torch.argmax(y_pred, dim=1)
            correct += (pred == y_batch).sum().item()
    exp.log_metric("train_loss", train_loss_sum / len(train_dataset), step=i)
    exp.log_metric("val_loss", test_loss_sum / len(test_dataset), step=i)
    exp.log_metric("val_accuracy", correct / len(test_dataset), step=i)
    print(f"Эпоха {i}: Train Loss: {train_loss_sum / len(train_dataset):.4f}, Val Loss: {test_loss_sum / len(test_dataset):.4f}, Val Acc: {correct / len(test_dataset):.4f}")

In [ ]:
exp.end()

## Бонусное задание: логгирование в WandB / Comet

Добавь логгирование метрик (например, loss и accuracy) во время обучения модели на MNIST с помощью `wandb` или `comet_ml`.

Что нужно сделать:
- зарегистрироваться в сервисе
- инициализировать логгер в ноутбуке
- отправлять метрики в каждом `epoch`
- сохранить конфигурацию эксперимента (hyperparams)